In [6]:
import numpy as np
import glob
import xml.etree.ElementTree as ET
from collections import defaultdict

PIE_PATH   = '/content/drive/MyDrive/PIE'
ATTR_PATH  = f'{PIE_PATH}/annotations_attributes'
OUTPUT_PATH= f'{PIE_PATH}/features'

# ── Load features and labels ───────────────────────────────
X = np.load(f'{OUTPUT_PATH}/X_features.npy')       # (N, 36) unscaled
y = np.load(f'{OUTPUT_PATH}/y_labels.npy')          # (N,) mental states

# Feature indices
F_DISTANCE_CHANGE_RATE = 26
F_CURRENT_SPEED        = 0
F_CROSSWALK            = 33
F_SPEED_VARIANCE       = 5
F_HEAD_TURN_FREQUENCY  = 13
F_LOOKS_LEFT           = 28
F_LOOKS_RIGHT          = 29
F_HESITATION_CYCLES    = 23
F_FORWARD_LEAN         = 18

LABEL_NAMES = {
    0:'Waiting', 1:'Hesitant',  2:'Committed',
    3:'Distracted', 4:'Aggressive', 5:'Jaywalk'
}

# ── Load PIE crossing labels ───────────────────────────────
# crossing=0  → will cross
# crossing=-1 → will not cross
print('Loading PIE crossing annotations...')
crossing_dict = {}
for f in glob.glob(f'{ATTR_PATH}/**/*.xml', recursive=True):
    root = ET.parse(f).getroot()
    for ped in root.findall('pedestrian'):
        pid      = ped.attrib.get('id','')
        crossing = ped.attrib.get('crossing', None)
        intention= ped.attrib.get('intention_prob', None)
        if pid and crossing is not None:
            crossing_dict[pid] = {
                'will_cross':  1 if int(crossing) == 0 else 0,
                'intention':   float(intention) if intention else None,
            }

print(f'Loaded {len(crossing_dict)} pedestrians')

# ── Compute crossing rate per mental state ─────────────────
print('\n' + '='*65)
print('EMPIRICAL VALUES FROM PIE DATASET')
print('These justify base risk values in compute_risk()')
print('='*65)

print(f'\n{"State":<14} {"n":>6}  {"cross_rate":>12}  {"base_risk_used":>15}')
print('-'*55)

base_risks_used = {
    0: 0.05,   # Waiting (no crosswalk)
    1: 0.40,   # Hesitant
    2: 0.85,   # Committed
    3: 0.25,   # Distracted
    4: 0.25,   # Aggressive (away from road)
    5: 0.80,   # Jaywalk
}

for cls in range(6):
    mask = y == cls
    n    = mask.sum()
    if n == 0:
        continue
    cross_rates = []
    intent_vals = []
    # For each frame in this class, look up crossing label
    # We use y_labels which is frame-level, but crossing is ped-level
    # So we approximate using the class crossing rate from PIE stats
    print(f'{LABEL_NAMES[cls]:<14} {n:>6}  '
          f'{"(see below)":>12}  '
          f'{base_risks_used[cls]:>15.2f}')

# ── Now compute actual stats from features ─────────────────
print('\n' + '='*65)
print('KINEMATIC EVIDENCE FOR INCREMENTS (+0.10, +0.15 etc)')
print('='*65)

for cls in range(6):
    mask = y == cls
    if mask.sum() == 0:
        continue

    X_cls = X[mask]

    toward_road  = (X_cls[:, F_DISTANCE_CHANGE_RATE] > 0)
    looking      = ((X_cls[:, F_LOOKS_LEFT] +
                     X_cls[:, F_LOOKS_RIGHT]) > 0)
    high_hesit   = (X_cls[:, F_HESITATION_CYCLES] > 0.5)
    at_crosswalk = (X_cls[:, F_CROSSWALK] == 1.0)
    fast         = (X_cls[:, F_CURRENT_SPEED] > 8.0)
    high_var     = (X_cls[:, F_SPEED_VARIANCE] > 200)
    fwd_lean     = (X_cls[:, F_FORWARD_LEAN] > 10)

    print(f'\n{LABEL_NAMES[cls]} (n={mask.sum()}):')
    print(f'  toward_road      : {toward_road.mean():.1%} of frames')
    print(f'  looking at road  : {looking.mean():.1%} of frames')
    print(f'  hesitation > 0.5 : {high_hesit.mean():.1%} of frames')
    print(f'  at crosswalk     : {at_crosswalk.mean():.1%} of frames')
    print(f'  speed > 8.0      : {fast.mean():.1%} of frames')
    print(f'  speed_var > 200  : {high_var.mean():.1%} of frames')
    print(f'  forward lean>10  : {fwd_lean.mean():.1%} of frames')
    print(f'  avg speed        : {X_cls[:,F_CURRENT_SPEED].mean():.3f}')
    print(f'  avg speed_var    : {X_cls[:,F_SPEED_VARIANCE].mean():.1f}')
    print(f'  avg dist_change  : {X_cls[:,F_DISTANCE_CHANGE_RATE].mean():.4f}')

# ── PIE crossing rates from attributes ────────────────────
print('\n' + '='*65)
print('PIE GROUND TRUTH CROSSING RATES (from annotations_attributes)')
print('='*65)

will_cross_list   = [v['will_cross'] for v in crossing_dict.values()]
intention_list    = [v['intention']  for v in crossing_dict.values()
                     if v['intention'] is not None]

print(f'\nOverall crossing rate : {np.mean(will_cross_list):.1%}')
print(f'Overall intention avg : {np.mean(intention_list):.3f}')
print(f'Intention distribution:')
bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.01]
for i in range(len(bins)-1):
    lo, hi = bins[i], bins[i+1]
    n = sum(1 for v in intention_list if lo <= v < hi)
    print(f'  [{lo:.1f} - {hi:.1f}) : {n:4d}  ({n/len(intention_list):.1%})')

print('\n' + '='*65)


Loading PIE crossing annotations...
Loaded 1842 pedestrians

EMPIRICAL VALUES FROM PIE DATASET
These justify base risk values in compute_risk()

State               n    cross_rate   base_risk_used
-------------------------------------------------------
Waiting         36836   (see below)             0.05
Hesitant         5409   (see below)             0.40
Committed       13294   (see below)             0.85
Distracted       6595   (see below)             0.25
Aggressive      14153   (see below)             0.25
Jaywalk          6504   (see below)             0.80

KINEMATIC EVIDENCE FOR INCREMENTS (+0.10, +0.15 etc)

Waiting (n=36836):
  toward_road      : 50.1% of frames
  looking at road  : 100.0% of frames
  hesitation > 0.5 : 73.1% of frames
  at crosswalk     : 11.3% of frames
  speed > 8.0      : 10.5% of frames
  speed_var > 200  : 0.0% of frames
  forward lean>10  : 30.3% of frames
  avg speed        : 3.228
  avg speed_var    : 11.0
  avg dist_change  : -0.0007

Hesitant (n=

In [7]:
import numpy as np
import glob
import xml.etree.ElementTree as ET
from collections import defaultdict
import os

PIE_PATH    = '/content/drive/MyDrive/PIE'
ATTR_PATH   = f'{PIE_PATH}/annotations_attributes'
LANDMARK_PATH = f'{PIE_PATH}/landmarks'
ANNOTATION_PATH = f'{PIE_PATH}/annotations'
OUTPUT_PATH = f'{PIE_PATH}/features'

SEQ_LEN  = 30
SEQ_STEP = 15

# ── Load crossing + intention per ped_id ───────────────────
crossing_dict = {}
for f in glob.glob(f'{ATTR_PATH}/**/*.xml', recursive=True):
    root = ET.parse(f).getroot()
    for ped in root.findall('pedestrian'):
        pid      = ped.attrib.get('id','')
        crossing = ped.attrib.get('crossing', None)
        intention= ped.attrib.get('intention_prob', None)
        if pid and crossing is not None:
            crossing_dict[pid] = {
                'will_cross': 1 if int(crossing) == 0 else 0,
                'intention':  float(intention) if intention else 0.5,
            }

# ── Rebuild sequence ped_id mapping ───────────────────────
sequence_ped_ids = []
sequence_labels  = []

y_sequences = np.load(f'{OUTPUT_PATH}/y_sequences.npy')

for ann_file in sorted(glob.glob(
        f'{ANNOTATION_PATH}/**/*.xml', recursive=True)):
    try:
        root = ET.parse(ann_file).getroot()
    except:
        continue

    parts    = ann_file.replace('\\','/').split('/')
    set_name = [p for p in parts if p.startswith('set')][0]
    vid_name = parts[-1].replace('_annt.xml','')

    for track in root.findall('track'):
        if track.attrib.get('label','') != 'pedestrian':
            continue

        ped_id = ''
        for box in track.findall('box'):
            attrs  = {a.attrib['name']: (a.text or '').strip()
                      for a in box.findall('attribute')}
            ped_id = attrs.get('id','')
            if ped_id: break
        if not ped_id: continue

        frames = []
        for box in track.findall('box'):
            if box.attrib.get('outside','0') == '1': continue
            fid     = int(box.attrib['frame'])
            lm_path = (f'{LANDMARK_PATH}/{set_name}/{vid_name}/'
                       f'frame_{fid:05d}.npy')
            if os.path.exists(lm_path):
                frames.append(fid)

        frames = sorted(frames)
        for start in range(0, len(frames)-SEQ_LEN+1, SEQ_STEP):
            if len(frames[start:start+SEQ_LEN]) == SEQ_LEN:
                sequence_ped_ids.append(ped_id)

n = min(len(sequence_ped_ids), len(y_sequences))
sequence_ped_ids = sequence_ped_ids[:n]

LABEL_NAMES = {
    0:'Waiting', 1:'Hesitant',  2:'Committed',
    3:'Distracted', 4:'Aggressive', 5:'Jaywalk'
}

# ── Compute crossing rate per mental state ─────────────────
print('='*65)
print('ACTUAL CROSSING RATES PER MENTAL STATE FROM PIE')
print('These are the correct base risk values')
print('='*65)

crossing_by_state  = defaultdict(list)
intention_by_state = defaultdict(list)

for i in range(n):
    pid   = sequence_ped_ids[i]
    state = int(y_sequences[i])
    if pid in crossing_dict:
        crossing_by_state[state].append(
            crossing_dict[pid]['will_cross']
        )
        intention_by_state[state].append(
            crossing_dict[pid]['intention']
        )

print(f'\n{"State":<14} {"n_seq":>6}  '
      f'{"cross_rate":>12}  {"avg_intention":>14}  '
      f'{"recommended_base":>18}')
print('-'*70)

recommended = {}
for cls in range(6):
    cr = crossing_by_state[cls]
    it = intention_by_state[cls]
    if len(cr) > 0:
        cross_rate  = np.mean(cr)
        avg_intent  = np.mean(it)
        # recommended base = average of both
        base        = round((cross_rate + avg_intent) / 2, 2)
        recommended[cls] = base
        print(f'{LABEL_NAMES[cls]:<14} {len(cr):>6}  '
              f'{cross_rate:>12.3f}  {avg_intent:>14.3f}  '
              f'{base:>18.3f}')

print('\n' + '='*65)
print('DATA-DRIVEN compute_risk() BASE VALUES')
print('='*65)
for cls in range(6):
    if cls in recommended:
        print(f'  {LABEL_NAMES[cls]:<14} → {recommended[cls]}')

print('\n' + '='*65)
print('DATA-DRIVEN INCREMENTS')
print('='*65)

X = np.load(f'{OUTPUT_PATH}/X_features.npy')
y = np.load(f'{OUTPUT_PATH}/y_labels.npy')

F_DISTANCE_CHANGE_RATE = 26
F_CURRENT_SPEED        = 0
F_CROSSWALK            = 33
F_SPEED_VARIANCE       = 5
F_HEAD_TURN_FREQUENCY  = 13
F_HESITATION_CYCLES    = 23
F_FORWARD_LEAN         = 18

for cls in range(6):
    mask  = y == cls
    X_cls = X[mask]
    if len(X_cls) == 0: continue

    toward = X_cls[:, F_DISTANCE_CHANGE_RATE] > 0
    hesit  = X_cls[:, F_HESITATION_CYCLES] > 0.5
    cross  = X_cls[:, F_CROSSWALK] == 1.0
    fast   = X_cls[:, F_CURRENT_SPEED] > 8.0
    hi_var = X_cls[:, F_SPEED_VARIANCE] > 200
    lean   = X_cls[:, F_FORWARD_LEAN] > 10

    base   = recommended.get(cls, 0.5)

    print(f'\n{LABEL_NAMES[cls]}  (base={base}):')

    # For each condition, show how much risk should increase
    # Increment = base × condition_rate (proportional to evidence)
    for cond_name, cond in [
        ('toward_road',   toward),
        ('hesit > 0.5',   hesit),
        ('at_crosswalk',  cross),
        ('speed > 8.0',   fast),
        ('speed_var>200', hi_var),
        ('fwd_lean > 10', lean),
    ]:
        rate = cond.mean()
        # increment = how often this condition occurs × 0.2 (max increment)
        increment = round(rate * 0.20, 2)
        print(f'  {cond_name:<18} occurs {rate:.1%}  '
              f'→ increment = {increment:.2f}')

ACTUAL CROSSING RATES PER MENTAL STATE FROM PIE
These are the correct base risk values

State           n_seq    cross_rate   avg_intention    recommended_base
----------------------------------------------------------------------
Waiting          1714         0.191           0.826               0.510
Hesitant          216         0.139           0.836               0.490
Committed         785         0.199           0.845               0.520
Distracted        218         0.151           0.899               0.530
Aggressive        616         0.172           0.853               0.510
Jaywalk           339         0.059           0.806               0.430

DATA-DRIVEN compute_risk() BASE VALUES
  Waiting        → 0.51
  Hesitant       → 0.49
  Committed      → 0.52
  Distracted     → 0.53
  Aggressive     → 0.51
  Jaywalk        → 0.43

DATA-DRIVEN INCREMENTS

Waiting  (base=0.51):
  toward_road        occurs 50.1%  → increment = 0.10
  hesit > 0.5        occurs 73.1%  → increment = 0.1